Here is how you can load the datasets with S3
1. In Config file: LOAD_FROM_S3=True
2. Enter you SSP Cloud secrets in a .env file (see READme)

In [3]:
#Imports
import s3fs
import pandas as pd
import logging
logging.basicConfig(level=logging.INFO)  #to be able to see the logging info+warnings(to only see warnings: logging.DEBUG)

from src.data_processing.data_load import data_loading
from src.data_processing.data_processing import DataProcessor


In [10]:
#To load data the from the different files directly
df_portefeuille, df_consommations, df_reclamations, df_interactions, df_impayes= data_loading("training")

PermissionError: Forbidden

In [3]:
#To load and process the data of a specific set (training, validation, evaluation) without normalisation of numerical values
processor = DataProcessor(mode="training")
df = processor.run(optional_normalisation=False)

INFO:src.data_processing.data_processing:Dropped columns from dataset: ['client_structure_familiale', 'client_nom_banque', 'client_nps_verbatim_n', 'client_nps_verbatim_n_moins1', 'impaye_nb_actions', 'impaye_montant_total', 'impaye_duree_max_action_jours', 'impaye_action_1er_impaye', 'impaye_action_1ere_lettre_relance', 'impaye_action_2eme_impaye', 'impaye_action_annonce_contentieux', 'impaye_action_en_recouvrement', 'impaye_action_mise_en_demeure', 'impaye_action_suspension_garanties', 'interaction_historique_mail']
INFO:src.data_processing.data_processing:Encoded with one-hot encoding: ['client_mode_paiement', 'client_frequence_paiement', 'courtier_reseau_courtage', 'courtier_segmentation_interne', 'courtier_type_commission', 'recla_canal_entrant_principale', 'recla_canal_entrant_secondaire']
INFO:src.data_processing.data_processing:Encoded with target encoding: ['client_ro_souscripteur', 'client_produit', 'client_garantie_base', 'client_garantie_base_plus_options', 'client_millesim

In [61]:
#Same but with normalisation of numerical values
processor = DataProcessor(mode="training")
df_norm = processor.run(optional_normalisation=True)

INFO:src.data_processing.data_processing:Dropped columns from dataset: ['client_structure_familiale', 'client_nps_verbatim_n', 'client_nps_verbatim_n_moins1', 'annee_mois_paiement', 'impaye_nb_actions', 'impaye_montant_total', 'impaye_duree_max_action_jours', 'impaye_action_1er_impaye', 'impaye_action_1ere_lettre_relance', 'impaye_action_2eme_impaye', 'impaye_action_annonce_contentieux', 'impaye_action_en_recouvrement', 'impaye_action_mise_en_demeure', 'impaye_action_suspension_garanties', 'interaction_historique_mail']
INFO:src.data_processing.data_processing:Encoded with target encoding: ['client_mode_paiement', 'client_frequence_paiement', 'courtier_reseau_courtage', 'courtier_type_commission', 'recla_canal_entrant_principale', 'recla_canal_entrant_secondaire']
INFO:src.data_processing.data_processing:Encoded with one-hot encoding: ['client_ro_souscripteur', 'client_produit', 'client_garantie_base', 'client_garantie_base_plus_options', 'client_millesime_tarifaire_produit', 'client_z

In [ ]:
df_norm #now for example, age is normalised (better to be processed by a linear model OR other specific models)

,client_age_souscripteur,client_ro_souscripteur,client_code_postal,client_departement,client_a_conjoint,client_nb_enfants,client_nb_assures_contrat,client_a_garantie_prevoyance,client_produit,client_garantie_base,...,recla_canal_entrant_principale_E-mail,recla_canal_entrant_principale_Formulaire web,recla_canal_entrant_principale_Réseaux sociaux,recla_canal_entrant_principale_Téléphone,recla_canal_entrant_principale_Valeur non référencée,recla_canal_entrant_secondaire_Courrier,recla_canal_entrant_secondaire_E-mail,recla_canal_entrant_secondaire_Formulaire web,recla_canal_entrant_secondaire_Réseaux sociaux,recla_canal_entrant_secondaire_Téléphone
0,2.040562,0.558597,0.447000,0.454435,0.0,0.0,-0.595937,0,-1.366055,-1.299895,...,0,0,0,0,0,0,0,0,0,0
1,1.919114,0.558597,-1.068101,-1.064775,0.0,0.0,-0.595937,1,-1.366055,-1.299120,...,0,0,0,0,0,0,0,0,0,0
2,1.494049,-1.844826,0.470326,0.454435,1.0,0.0,0.706991,0,-1.366055,-1.299895,...,0,0,0,0,0,0,0,0,0,0
3,1.919114,-1.844826,0.443572,0.454435,0.0,0.0,-0.595937,0,-1.366055,-1.299120,...,0,0,0,0,0,0,0,0,0,0
4,1.615496,-1.844826,0.447000,0.454435,1.0,0.0,0.706991,1,-1.366055,-1.299120,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132571,-1.663582,0.558597,0.625123,0.631087,0.0,0.0,-0.595937,0,-1.223350,-1.120666,...,0,0,0,0,0,0,0,0,0,0
132572,-0.024043,0.558597,-1.378650,-1.382749,0.0,0.0,-0.595937,0,-1.488218,-1.443664,...,0,0,0,0,0,0,0,0,0,0
132573,-2.331542,0.558597,0.627244,0.631087,0.0,0.0,-0.595937,0,-1.223350,-1.295253,...,0,0,0,0,0,0,0,0,0,0
132574,-2.513713,0.558597,1.440634,1.443687,0.0,0.0,-0.595937,1,-0.795205,-0.697177,...,0,0,0,0,0,0,0,0,0,0


- The processing pipeline takes time (40 sec)
- **Alternative option**: load the already merged dataset (in parquet format) stored in S3, in the group-2 bucket "projet-bdc-alptis-g2" (around 10 sec)

In [3]:
#Use s3fs library to load the bucket where the parquet is stored
import Config 

fs = s3fs.S3FileSystem(
        endpoint_url='https://' + Config.S3_ENDPOINT,
        key=Config.S3_ACCESS_KEY,
        secret=Config.S3_SECRET_KEY,
        token=Config.S3_SESSION_TOKEN,
        client_kwargs={'verify': Config.S3_VERIFY_SSL}
    )

#Load the merged file (stored in parquet format) in the bucket
with fs.open(Config.MERGED_PARQUET_S3, "rb") as f:
    df_merged= pd.read_parquet(f)